In [1]:
import random

import pandas as pd

import hypex
from hypex import ABTest
from hypex.dataset import Dataset, TargetRole, TreatmentRole

In [2]:
spark = hypex.dataset.backends.spark_backend.SparkNavigation._get_spark_session(mode="local")

your 131072x1 screen size is bogus. expect trouble
25/10/23 14:38:23 WARN Utils: Your hostname, DESKTOP-Q62I9LF resolves to a loopback address: 127.0.1.1; using 192.168.97.32 instead (on interface eth0)
25/10/23 14:38:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/23 14:38:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = Dataset(
    roles={
        # "user_id": InfoRole(int),
        "treat": TreatmentRole(),
        # "pre_spends": FeatureRole(),
        "post_spends": TargetRole(),
        # "gender": FeatureRole()
    }, data="data.csv",
    session=spark
)
# data

In [4]:
data

DataFrame[user_id: int, signup_month: int, treat: int, pre_spends: double, post_spends: double, age: double, gender: string, industry: string]

In [5]:
spark.stop()

In [6]:
df = pd.read_csv("data.csv")
types = df.dtypes

In [7]:
for col, dtype in types.items():
    print(col, dtype)

user_id int64
signup_month int64
treat int64
pre_spends float64
post_spends float64
age float64
gender object
industry object


In [8]:
data["treat"] = [random.choice([0, 1, 2]) for _ in range(len(data))]
data

TypeError: 'NoneType' object cannot be interpreted as an integer

The roles' data types can be assigned automatically as shown below. Also, the fields, which were not marked, receive Feature role by default.

In [ ]:
data.roles

{'user_id': Info(<class 'int'>),
 'treat': Treatment(<class 'int'>),
 'pre_spends': Feature(<class 'float'>),
 'post_spends': Target(<class 'float'>),
 'gender': Feature(<class 'str'>),
 'signup_month': Default(<class 'int'>),
 'age': Default(<class 'float'>),
 'industry': Default(<class 'str'>)}

## AB test
Then we select one of the pre-assembled pipelines, in our case `ABTest`. Also, a custom pipeline can be created based on your specific needs and requirements with custom executors.
After that we wrap our prepared `dataset` into `ExperimentData` to be able to run experiments on it and then execute the test with this data passed as the argument.

In [ ]:
test = ABTest()
result = test.execute(data)

### Experiment results
To show the report with summary of the test we run the `resume` method of the output of the experiment.

It displays the results of the test in the form of a table with the following columns:
- `feature`: name of the target feature, change of which we want to analyze.
- `group`: name of the test group we compare with the control group.
- `TTest pass`: result of the TTest, if it is significant or not.
- `TTest p-value`: p-value of the TTest shows the probability of obtaining the result when the null hypothesis is true. The lower the value the more significant the result is.
- `control mean`: the mean of the feature value across the control group.
- `test mean`: the mean of the feature value across the test group.
- `difference`: the difference between the mean of the test group and the mean of the control group.
- `difference %`: the normalized difference between the mean of the test group and the mean of the control group.

In [ ]:
result.resume

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value
0,post_spends,1,452.386765,451.363942,-1.022823,-0.226095,NOT OK,0.283005
1,post_spends,2,452.386765,452.751362,0.364597,0.080594,NOT OK,0.707373


The method sizes shows the statistics on the groups of the data.

The columns are:
- `control size`: the size of the control group.
- `test size`: the size of the test group.
- `control size %`: the share of the control group in the whole dataset.
- `test size %`: the share of the test group in the whole dataset.
- `group`: name of the test group.

In [ ]:
result.sizes

,control size,test size,control size %,test size %,group
1,3400,3336,50,49,1
2,3400,3264,51,48,2


In [ ]:
result.multitest

,field,test,old p-value,new p-value,correction,rejected,group
0,post_spends,TTest,0.283005,0.566011,0.5,False,1
1,post_spends,TTest,0.707373,0.707373,1.0,False,2


## Additional tests in AB Test 

It is possible to add u-test and chi2-test in pipeline.

In [ ]:
test = ABTest(additional_tests=['t-test', 'u-test', 'chi2-test'])
result = test.execute(data)

The additional columns are:
- `UTest pass`: result of the UTest, if it is significant or not.
- `UTest p-value`: p-value of the UTest shows the probability of obtaining the result when the null hypothesis is true. The lower the value the more significant the result is.
- `Chi2Test pass`: result of the Chi2Test, if it is significant or not.
- `Chi2Test p-value`: p-value of the Chi2Test shows the probability of obtaining the result when the null hypothesis is true. The lower the value the more significant the result is.

In [ ]:
result.resume

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value,UTest pass,UTest p-value
0,post_spends,1,452.386765,451.363942,-1.022823,-0.226095,NOT OK,0.283005,NOT OK,0.190650
1,post_spends,2,452.386765,452.751362,0.364597,0.080594,NOT OK,0.707373,NOT OK,0.620935


In [ ]:
result.multitest

,field,test,old p-value,new p-value,correction,rejected,group
0,post_spends,TTest,0.283005,0.849016,0.333333,False,1
1,post_spends,TTest,0.707373,1.000000,0.707373,False,2
2,post_spends,UTest,0.190650,0.762598,0.250000,False,1
3,post_spends,UTest,0.620935,1.000000,0.620935,False,2


In [ ]:
result.sizes

,control size,test size,control size %,test size %,group
1,3400,3336,50,49,1
2,3400,3264,51,48,2


## ABn Test 

Finally, we may run multiple ab tests with different methods.

In [ ]:
test = ABTest(multitest_method="bonferroni")
result = test.execute(data)

In [ ]:
result.resume

,feature,group,control mean,test mean,difference,difference %,TTest pass,TTest p-value
0,post_spends,1,452.386765,451.363942,-1.022823,-0.226095,NOT OK,0.283005
1,post_spends,2,452.386765,452.751362,0.364597,0.080594,NOT OK,0.707373


In [ ]:
result.sizes

,control size,test size,control size %,test size %,group
1,3400,3336,50,49,1
2,3400,3264,51,48,2


In [ ]:
result.multitest

,field,test,old p-value,new p-value,correction,rejected,group
0,post_spends,TTest,0.283005,0.566011,0.5,False,1
1,post_spends,TTest,0.707373,0.707373,1.0,False,2
